# Dataset (https://www.kaggle.com/datasets/kazanova/sentiment140)

* https://www.youtube.com/watch?v=mY-lIkjrvzI&list=PL049bGjkT7dIARLPrxvA4-tbEcg6k0UDX&index=123

In [1]:
from google.colab import drive
drive.mount('/gdrive')

Mounted at /gdrive


In [2]:
folder = '/gdrive/MyDrive/deep_learning_course/data/tweets/'

In [3]:
!ls '/gdrive/MyDrive/deep_learning_course/data/tweets/'

ls: cannot access '/gdrive/MyDrive/deep_learning_course/data/tweets/': No such file or directory


In [4]:
import pandas as pd

In [5]:
df = pd.read_csv(folder + "training.1600000.processed.noemoticon.csv", encoding="latin", header=None)

FileNotFoundError: [Errno 2] No such file or directory: '/gdrive/MyDrive/deep_learning_course/data/tweets/training.1600000.processed.noemoticon.csv'

In [ ]:
df.head()

In [ ]:
df = df.iloc[:,[0, 5]]

In [ ]:
df.head()

In [ ]:
df.columns = ['sentiment', 'tweet']

In [ ]:
df.head()

In [ ]:
df['sentiment'].value_counts()

In [ ]:
sents = {0:"negatif", 4:"positif"}

In [ ]:
sents[0]

In [ ]:
df['sentiment'] = df['sentiment'].replace(sents)

In [ ]:
df.head()

In [ ]:
df.sample(10)

# Preprocessing

In [ ]:
import re

In [ ]:
text_cleaning_regex = "@\S+|https?:\S+|http?:\S+|[^A-Za-z0-9]+"

In [ ]:
import nltk

In [ ]:
nltk.download('stopwords')

In [ ]:
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

In [ ]:
english_stopwords = stopwords.words('english')

In [ ]:
english_stopwords

In [ ]:
stemmer = SnowballStemmer('english')

In [ ]:
stemmer.stem('cats')

In [ ]:
def preprocess(text):

  # les mentions les liens et tout ce qui n'est pas alphanumérique
  text_cleaning_regex = "@\S+|https?:\S+|http?:\S+|[^A-Za-z0-9]+"
  text = re.sub(text_cleaning_regex, " ", str(text).lower().strip())

  tokens = []
  # stopwords and stem
  for word in text.split(" "):
    if word not in english_stopwords:
      word_stem = stemmer.stem(word)
      tokens.append(word_stem)

  cleaned_text = " ".join(tokens).strip()


  return cleaned_text

In [ ]:
preprocess("@kevin_degila Hello, I was wondering when your course on deep learning would be available ? http://kevindegila.com")

In [ ]:
df['text'] = df['tweet'].apply(preprocess)

In [ ]:
df.head()

# Train test split

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train_data, test_data = train_test_split(df, test_size=0.2, stratify=df['sentiment'], random_state=42)

In [ ]:
len(train_data)

In [ ]:
len(test_data)

In [ ]:
test_data['sentiment'].value_counts()

In [ ]:
def length(text):
  return len(text.split(' '))

In [ ]:
df['len'] = df['text'].apply(length)

In [ ]:
df['len'].min(), df['len'].max(), df['len'].mean(), df['len'].median()

In [ ]:
import seaborn as sns
sns.displot(df['len'])

In [ ]:
df['len'].describe()

# Tokenization and text to sequences

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
vocab_size = 70000
maxlen = 15
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(train_data['text'])
word_index = tokenizer.word_index

training_sequences = tokenizer.texts_to_sequences(train_data['text'])
training_padded = pad_sequences(training_sequences, padding="post", maxlen=maxlen, truncating="post")
test_sequences = tokenizer.texts_to_sequences(test_data['text'])
test_padded = pad_sequences(test_sequences, padding="post", maxlen=15, truncating="post")

In [ ]:
training_padded

# Label Encoder

In [ ]:
train_data['sentiment']

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
encoder = LabelEncoder()
training_labels = encoder.fit_transform(train_data['sentiment'])

In [ ]:
training_labels

In [ ]:
test_data.shape

In [ ]:
test_labels = encoder.transform(test_data['sentiment'])

In [ ]:
training_labels.shape

In [ ]:
training_labels = training_labels.reshape(-1, 1)
test_labels = test_labels.reshape(-1, 1)

In [ ]:
test_labels.shape

# Modeling

In [ ]:
import tensorflow as tf

In [ ]:
import numpy as np

In [ ]:
np.power(70000, 1/4)

In [ ]:
embedding_dim = 16
model = tf.keras.models.Sequential(
    [
        tf.keras.layers.Embedding(vocab_size, embedding_dim),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dense(8, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ]
)

model.compile(loss='binary_crossentropy', optimizer="adam", metrics=['accuracy'])
model_ckp = tf.keras.callbacks.ModelCheckpoint(filepath="best_model.h5",
                            monitor="val_accuracy",
                            mode="max",
                            save_best_only=True)
stop = tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True)
h = model.fit(training_padded, training_labels, epochs=50, batch_size=2048,
              validation_data=(test_padded, test_labels),
              callbacks=[model_ckp])

In [ ]:
import matplotlib.pyplot as plt


def plot_graphs(history, string):
    plt.plot(history.history[string])
    plt.plot(history.history['val_'+string])
    plt.xlabel("Epochs")
    plt.ylabel(string)
    plt.legend([string, 'val_'+string])
    plt.show()

In [ ]:
plot_graphs(h, 'accuracy')
plot_graphs(h, "loss")

# Recurrent Neural Networks

In [ ]:
"J'étais content, maintenant je suis enervé"  negatif
"J'étais enervé, maintenant je suis content"   positif

In [ ]:
embedding_dim = 16
model = tf.keras.models.Sequential(
    [
        tf.keras.layers.Embedding(vocab_size, embedding_dim),
        tf.keras.layers.SimpleRNN(10),
        tf.keras.layers.Dense(8, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ]
)

model.compile(loss='binary_crossentropy', optimizer="adam", metrics=['accuracy'])
model_ckp = tf.keras.callbacks.ModelCheckpoint(filepath="best_model.h5",
                            monitor="val_accuracy",
                            mode="max",
                            save_best_only=True)
stop = tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True)

In [ ]:
model.summary()

In [ ]:
h = model.fit(training_padded, training_labels, epochs=50, batch_size=2048,
              validation_data=(test_padded, test_labels),
              callbacks=[model_ckp])

In [ ]:
plot_graphs(h, 'accuracy')
plot_graphs(h, "loss")

# LSTM

In [ ]:
Je vis en , J'ai vécu dans différentes villes jusque-là, sauf la capital PARIS

In [ ]:
adam = tf.keras.optimizers.Adam()

In [ ]:
embedding_dim = 16
model = tf.keras.models.Sequential(
    [
        tf.keras.layers.Embedding(vocab_size, embedding_dim),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(16, return_sequences=True, dropout=0.25)),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(16)),
        tf.keras.layers.Dense(8, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ]
)

model.compile(loss='binary_crossentropy', optimizer="adam", metrics=['accuracy'])
model_ckp = tf.keras.callbacks.ModelCheckpoint(filepath="best_model.h5",
                            monitor="val_accuracy",
                            mode="max",
                            save_best_only=True)
stop = tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True)

In [ ]:
model.summary()

In [ ]:
h = model.fit(training_padded, training_labels, epochs=50, batch_size=2048,
              validation_data=(test_padded, test_labels),
              callbacks=[model_ckp])

In [ ]:
plot_graphs(h, 'accuracy')
plot_graphs(h, "loss")

# Transfert Learning

In [ ]:
glove_file = folder + "glove.6B.50d.txt"

In [ ]:
glove_file

In [ ]:
glove_embeddings = {}

In [ ]:
with open(glove_file) as f:
  for line in f:

    values = line.split(" ")
    word = values[0]
    vector = np.asarray(values[1:], dtype="float32")
    glove_embeddings[word] = vector


In [ ]:
glove_embeddings

In [ ]:
glove_embeddings['the']

In [ ]:
len(glove_embeddings)

In [ ]:
words = word_index.keys()

In [ ]:
i = 0
for word in words:
  if glove_embeddings.get(word) is not None:
    i = i + 1
print(i)

In [ ]:
len(words)

In [ ]:
embedding_matrix = np.zeros((vocab_size, 50))

In [ ]:
embedding_matrix.shape

In [ ]:
embedding_matrix[0]

In [ ]:
len(word_index)

In [ ]:
for word, i in word_index.items():
  if i > vocab_size -1:
    break
  embedding_vector = glove_embeddings.get(word)
  if embedding_vector is not None:
    embedding_matrix[i] = embedding_vector


In [ ]:
embedding_matrix[5]

In [ ]:
embedding_dim = 16
model = tf.keras.models.Sequential(
    [
        tf.keras.layers.Embedding(vocab_size, 50, weights=[embedding_matrix], trainable=False),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(16, return_sequences=True, dropout=0.25)),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(16)),
        tf.keras.layers.Dense(8, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ]
)

model.compile(loss='binary_crossentropy', optimizer="adam", metrics=['accuracy'])
model_ckp = tf.keras.callbacks.ModelCheckpoint(filepath="best_model.h5",
                            monitor="val_accuracy",
                            mode="max",
                            save_best_only=True)
stop = tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=2, restore_best_weights=True)

In [ ]:
h = model.fit(training_padded, training_labels, epochs=50, batch_size=2048,
              validation_data=(test_padded, test_labels),
              callbacks=[model_ckp])

In [ ]:
plot_graphs(h, 'accuracy')
plot_graphs(h, "loss")